In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
import os
import re
import time

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0.1
)

print("✓ Modelo configurado para Chain-of-Thought")
print("✓ Temperature baja para razonamiento consistente")

✓ Modelo configurado para Chain-of-Thought
✓ Temperature baja para razonamiento consistente


In [2]:
def few_shot_cot():
    print("=== FEW-SHOT CHAIN-OF-THOUGHT ===")

    nuevo_problema = "Hoy registramos 3% de mortalidad en el centro Ensenada. ¿Es un valor normal y qué acciones debemos tomar?"

    prompt = r"""Resuelve la problemática de producción acuícola mostrando el razonamiento paso a paso:

Problema: El centro Huito registró un FCR de 1.4 esta semana con salmón Atlántico de 3kg promedio. ¿Es eficiente?
Razonamiento:
1. El FCR (Factor de Conversión Alimenticia) mide cuántos kg de alimento se necesitan para producir 1kg de biomasa.
2. Un FCR de 1.4 significa que se usaron 1.4kg de alimento por cada 1kg de crecimiento.
3. Para salmón Atlántico en etapa de engorda, un FCR eficiente se considera entre 1.1 y 1.3.
4. Un FCR de 1.4 está por encima del rango óptimo, lo que indica un uso menos eficiente del alimento.
5. Posibles causas: temperatura del agua elevada, estado sanitario del pez, calidad del pellet o sobrealimentación.
Respuesta: El FCR de 1.4 no es óptimo. Se recomienda revisar la temperatura del agua, el estado sanitario y ajustar las raciones de alimento para mejorar la eficiencia.

Problema: El centro Puelche reporta un peso promedio de 2.8kg con una desviación alta entre jaulas. ¿Qué indica esto?
Razonamiento:
1. Una alta desviación entre jaulas indica que los peces no están creciendo de forma homogénea.
2. Esto puede deberse a diferencias en densidad de carga entre jaulas, distribución desigual del alimento o presencia de peces dominantes.
3. Una desviación alta afecta la planificación de cosecha porque no todas las jaulas alcanzan el peso objetivo al mismo tiempo.
4. Se debe realizar una biometría detallada por jaula para identificar cuáles están rezagadas.
5. Adicionalmente, revisar los sistemas de alimentación para asegurar una distribución uniforme del pellet.
Respuesta: La alta desviación indica crecimiento no homogéneo. Se recomienda biometría por jaula, revisar densidades y sistemas de alimentación.

Problema: """ + nuevo_problema + r"""
Razonamiento:"""

    start_time = time.time()
    response_text = ""

    try:
        for chunk in llm.stream([HumanMessage(content=prompt)]):
            print(chunk.content, end="", flush=True)
            response_text += chunk.content
            time.sleep(0.03)

        end_time = time.time()
        print(f"\n\n[Streaming completado en {end_time - start_time:.2f} segundos]")

        tiene_pasos_numerados = bool(re.search(r'\d+\.', response_text))
        tiene_respuesta_final = 'respuesta' in response_text.lower()

        print("\n=== ANÁLISIS DEL PATRÓN ===")
        print(f"✓ Pasos numerados: {'Sí' if tiene_pasos_numerados else 'No'}")
        print(f"✓ Respuesta final clara: {'Sí' if tiene_respuesta_final else 'No'}")

    except Exception as e:
        print(f"Error: {e}")

few_shot_cot()

=== FEW-SHOT CHAIN-OF-THOUGHT ===
11.. ** **DefDefinirinir el el contexto contexto de de mortal mortalidadidad normal normal****:: En En la la acu acuiciculturaultura,, los los niveles niveles acept aceptablesables de de mortal mortalidadidad var varíanían según según la la especie especie,, la la etapa etapa de de crecimiento crecimiento y y las las condiciones condiciones ambientales ambientales.. Para Para el el sal salmmónón Atl Atlántánticoico en en etapa etapa de de eng engordaorda,, una una mortal mortalidadidad diaria diaria normal normal suele suele estar estar por por debajo debajo del del  00..55%.

%.

22.. ** **ComparCompararar el el valor valor registrado registrado con con el el rango rango normal normal****:: Un Un  33%% de de mortal mortalidadidad diaria diaria es es significativamente significativamente más más alto alto que que el el rango rango acept aceptableable (< (<00..55%),%), lo lo que que indica indica un un problema problema grave grave que que debe debe abo